# 2 pamoka – Microsoft Agent Framework tyrinėjimas

**Microsoft Agent Framework (MAF)** yra vieningas pagrindas kuriant DI agentus. Jis suteikia aiškią, sudedamą architektūrą su keturiais pagrindiniais blokais:

- **Klientas** – jungiasi prie DI modelio taško ir tvarko komunikaciją
- **Agentas** – apgaubia klientą su instrukcijomis ir įrankių apibrėžimais
- **Įrankiai** – praplečia agento galimybes su vartotojo funkcijomis, kurias gali kviesti modelis
- **Sesija** – palaiko pokalbio istoriją kelių žinučių sąveikai

Šioje pamokoje kursime **kelionių užsakymo agentą**, kuris patikrina kelionės vietos prieinamumą naudodamas šias sąvokas.


## Nustatymas


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Agentų sistemos architektūros supratimas

Microsoft Agent Framework naudoja sluoksniuotą architektūrą:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klientas** – `FoundryChatClient` jungiasi prie Azure OpenAI diegimo. Jis tvarko autentifikavimą, užklausų formatavimą ir atsakymų analizę.
2. **Agentas** – Sukurtas iš kliento naudojant `provider.create_agent()`, agentas sujungia modelio prieigą su instrukcijomis (sistemos užklausa) ir įrankiais.
3. **Įrankiai** – Python funkcijos, pažymėtos `@tool`, kurias agentas gali iškviesti atlikti veiksmus arba gauti duomenis.
4. **Seansas** – `AgentSession` objektas (sukurtas naudojant `agent.create_session()`), saugantis pokalbių istoriją, leidžiantis vykdyti daugiakartį dialogą, kur agentas prisimena ankstesnį kontekstą.

Kuriame kiekvieną sluoksnį žingsnis po žingsnio.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Priemonių pridėjimas naudojant @tool dekoratorių

Priemonės leidžia agentams atlikti veiksmus, neapsiribojant tekstų generavimu. `@tool` dekoratorius paverčia paprastą Python funkciją į ką nors, ką agentas gali iškviesti.

Pagrindiniai punktai:
- Naudokite `Annotated[type, "description"]`, kad modelis suprastų kiekvieną parametrą.
- Docstring tampa priemonės aprašymu, kurį mato modelis.
- `approval_mode="never_require"` reiškia, kad priemonė veikia automatiškai be vartotojo patvirtinimo.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agentas su įrankiais kūrimas

Dabar sujungiame klientą, instrukcijas ir įrankius į agentą. `instructions` veikia kaip sistemos užklausa — jie apibrėžia agente asmenybę ir elgesį.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Daugiau posūkių pokalbiai su seansais

`AgentSession` (sukurtas per `agent.create_session()`) seka visus pranešimus pokalbio metu. Perdami tą patį seansą kiekvienam `agent.run()` kvietimui, agentas turi prieigą prie visos pokalbio istorijos ir gali grįžti prie ankstesnių pranešimų.

Mes perduodame `tools=[check_destination_availability]`, kad agentas kiekvieno posūkio metu galėtų naudoti mūsų prieinamumo tikrintuvą.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Santrauka

Šioje pamokoje jūs ištyrėte keturis Microsoft Agent Framework pagrindus:

| Sąvoka | Ką Išmokote |
|---------|------------------|
| **Klientas** | `FoundryChatClient` jungiasi prie Azure OpenAI su autentifikacija pagal kredencialus |
| **Agentas** | `provider.create_agent()` sujungia modelio ryšį su instrukcijomis ir vardu |
| **Įrankiai** | `@tool` dekoratorius leidžia agentui kviesti Python funkcijas |
| **Seansas** | `agent.create_session()` palaiko pokalbio istoriją per kelis apsisukimus |

Šie pagrindiniai elementai kartu sudaro agentus, kurie gali vesti natūralias pokalbes, kviesti išorines funkcijas ir palaikyti kontekstą — tai pamatas pažangesniems agentiniams modeliams vėlesnėse pamokose.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
